## V3: Auto Loader Pipeline (Single Notebook)

This pipeline uses **Auto Loader** (`cloudFiles`) for incremental ingestion.

**Key differences from V1 and V2:**
* **Bronze**: Auto Loader streams new files incrementally — only processes **new** files since the last run (no manual dedup needed)
* **Silver/Gold**: Automatic dependency management via `dp.read()`
* **Single notebook**: All layers defined in one place, pipeline resolves the DAG

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType

VOLUME_PATH = "/Volumes/gharchive_dev/v3_autoloader/raw_files/"

# Explicit schema for GH Archive events
# payload stored as STRING (raw JSON) to avoid schema merge conflicts across event types
GHARCHIVE_SCHEMA = StructType([
    StructField("id", StringType()),
    StructField("type", StringType()),
    StructField("actor", StructType([
        StructField("id", LongType()),
        StructField("login", StringType()),
        StructField("display_login", StringType()),
        StructField("gravatar_id", StringType()),
        StructField("url", StringType()),
        StructField("avatar_url", StringType()),
    ])),
    StructField("repo", StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
        StructField("url", StringType()),
    ])),
    StructField("payload", StringType()),
    StructField("public", BooleanType()),
    StructField("created_at", StringType()),
    StructField("org", StructType([
        StructField("id", LongType()),
        StructField("login", StringType()),
        StructField("gravatar_id", StringType()),
        StructField("url", StringType()),
        StructField("avatar_url", StringType()),
    ])),
])


@dp.table(
    name="gharchive_bronze",
    comment="Raw GitHub Archive events - incrementally ingested via Auto Loader"
)
def gharchive_bronze():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("rescuedDataColumn", "_rescued_data")
        .schema(GHARCHIVE_SCHEMA)
        .load(VOLUME_PATH)
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
    )

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, explode_outer, to_timestamp, get_json_object, from_json
from pyspark.sql.types import ArrayType, StructType, StructField, StringType


@dp.table(
    name="gharchive_silver",
    comment="Flattened GitHub Archive events with exploded commits"
)
def gharchive_silver():
    df = dp.read("gharchive_bronze")

    # Flatten repo struct
    if "repo" in df.columns:
        df = (df
            .withColumn("repo_id", col("repo.id"))
            .withColumn("repo_name", col("repo.name"))
            .withColumn("repo_url", col("repo.url"))
        )

    # Flatten actor struct
    if "actor" in df.columns:
        df = (df
            .withColumn("actor_id", col("actor.id"))
            .withColumn("actor_login", col("actor.login"))
            .withColumn("actor_display_login", col("actor.display_login"))
            .withColumn("actor_avatar_url", col("actor.avatar_url"))
        )

    # Extract commit fields from payload JSON string
    # payload is stored as raw JSON string in bronze to avoid schema merge conflicts
    COMMIT_SCHEMA = ArrayType(StructType([
        StructField("sha", StringType()),
        StructField("message", StringType()),
        StructField("author", StructType([
            StructField("name", StringType()),
            StructField("email", StringType()),
        ])),
    ]))

    commits_json = get_json_object(col("payload"), "$.commits")
    df = df.withColumn("_commits", from_json(commits_json, COMMIT_SCHEMA))

    # Explode commits (only PushEvents have them, others will be null)
    df = df.withColumn("commit", explode_outer("_commits"))
    df = (df
        .withColumn("commit_sha", col("commit.sha"))
        .withColumn("commit_message", col("commit.message"))
        .withColumn("commit_author_name", col("commit.author.name"))
        .withColumn("commit_author_email", col("commit.author.email"))
    )
    df = df.drop("commit", "_commits")

    # Extract action from payload (common across event types)
    df = df.withColumn("payload_action", get_json_object(col("payload"), "$.action"))

    df = df.withColumn("created_at", to_timestamp("created_at"))

    return df.drop("repo", "actor", "payload", "org", "_rescued_data")

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import date_trunc, count, countDistinct


@dp.table(
    name="gharchive_gold_activity_counts",
    comment="Hourly GitHub event counts by event type"
)
def gharchive_gold_activity_counts():
    df = dp.read("gharchive_silver")

    return (
        df
        .withColumn("event_hour", date_trunc("hour", "created_at"))
        .groupBy("event_hour", "type")
        .agg(
            count("*").alias("event_count"),
            countDistinct("actor_login").alias("unique_actors"),
            countDistinct("repo_name").alias("unique_repos")
        )
        .orderBy("event_hour", "type")
    )

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import count, countDistinct, col


@dp.table(
    name="gharchive_gold_top_repos",
    comment="Top repositories ranked by total event count"
)
def gharchive_gold_top_repos():
    df = dp.read("gharchive_silver")

    return (
        df
        .groupBy("repo_name")
        .agg(
            count("*").alias("total_events"),
            countDistinct("actor_login").alias("unique_contributors"),
            countDistinct("type").alias("event_types")
        )
        .orderBy(col("total_events").desc())
    )

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import count, countDistinct, min, max, col


@dp.table(
    name="gharchive_gold_top_actors",
    comment="Top GitHub actors ranked by contribution count"
)
def gharchive_gold_top_actors():
    df = dp.read("gharchive_silver")

    return (
        df
        .groupBy("actor_login")
        .agg(
            count("*").alias("total_events"),
            countDistinct("repo_name").alias("unique_repos"),
            countDistinct("type").alias("event_types"),
            min("created_at").alias("first_event"),
            max("created_at").alias("last_event")
        )
        .orderBy(col("total_events").desc())
    )